In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

BASE      = next(p for p in [Path().resolve()] + list(Path().resolve().parents) if (p / "DECISIONS.md").exists())
HORAS_IN  = BASE / "data/processed/horas_por_setor_2023_corrigido.csv"
PIB_IN    = BASE / "data/processed/pib_setorial_2012_2023.csv"
BASE_OUT  = BASE / "data/processed/base_analitica_setorial_2023.csv"
TAB_OUT   = BASE / "outputs/tables/cenario_base_setorial.csv"
DIR_FIGS  = BASE / "outputs/figures"

plt.rcParams.update({
    "figure.dpi": 150,
    "font.family": "sans-serif",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
})
print("Imports OK")

In [ ]:
df_horas = pd.read_csv(HORAS_IN, index_col=0)
df_pib   = pd.read_csv(PIB_IN)

pib_2023     = df_pib[df_pib["ano"] == 2023].squeeze()
pib_total_rs = pib_2023["pib_total"] * 1e6

assert (df_horas["pct_abaixo_40"] + df_horas["pct_40h_exatas"] +
        df_horas["pct_41_44h"] + df_horas["pct_acima_44h"]).between(99, 101).all()
print("✓ Validação das faixas OK")
print(f"PIB total 2023: R$ {pib_total_rs/1e12:.2f} tri")

In [ ]:
SEMANAS_ANO = 52

horas_faixa = np.array([41, 42, 43, 44])
DELTA_H_C   = ((40 - horas_faixa) / horas_faixa).mean()

print(f"\nFormulação C adotada (ver DECISIONS.md D16):")
print(f"  ΔH = média[ ΔH(41), ΔH(42), ΔH(43), ΔH(44) ] = {DELTA_H_C*100:.4f}%")
print(f"  (Formulação A para referência: {(40-42)/42*100:.4f}%)")
print(f"  Viés A vs C: 18,14% → limiar > 15% → C adotada")

correspondencia = [
    ("Agropecuária",          "agropecuaria",          True,  "direta"),
    ("Indústria geral",       "ind_transformacao",     True,  "parcial"),
    ("Construção",            "construcao",            True,  "direta"),
    ("Comércio e rep.",       "comercio",              True,  "direta"),
    ("Transp. e armaz.",      "transporte",            True,  "direta"),
    ("Inf. e serv. prof.",    "info_comunicacao",      True,  "parcial"),
    ("Adm. públ. e educação", "adm_publica_saude_educ",True, "ampla"),
    ("Alojamento e alim.",    "outros_servicos",       False, "muito_imperfeita"),
    ("Saúde",                 "adm_publica_saude_educ",False, "nao_separavel"),
    ("Serv. domésticos",      "outros_servicos",       False, "muito_imperfeita"),
]

In [ ]:
registros = []

for setor, col, incluir, qualidade in correspondencia:
    if setor not in df_horas.index:
        continue

    n     = df_horas.loc[setor, "n"]
    h_med = df_horas.loc[setor, "media"]
    h_mdn = df_horas.loc[setor, "mediana"]
    p4144 = df_horas.loc[setor, "pct_41_44h"]
    p_44  = df_horas.loc[setor, "pct_acima_44h"]
    p_40  = df_horas.loc[setor, "pct_40h_exatas"]
    p_sub = df_horas.loc[setor, "pct_abaixo_40"]

    vab_rs          = pib_2023[col] * 1e6
    total_horas_ano = n * h_med * SEMANAS_ANO
    produt_hora     = vab_rs / total_horas_ano

    parcela_afetada   = p4144 / 100.0
    delta_horas_setor = parcela_afetada * DELTA_H_C   # Formulação C
    delta_vab_rs      = vab_rs * delta_horas_setor
    intensidade_pct   = delta_horas_setor * 100

    registros.append({
        "setor":             setor,
        "col_cnt":           col,
        "qualidade_proxy":   qualidade,
        "incluir":           incluir,
        "n_ocupados":        int(n),
        "h_media":           round(h_med, 2),
        "h_mediana":         round(h_mdn, 1),
        "pct_abaixo_40":     round(p_sub, 1),
        "pct_40h_exatas":    round(p_40, 1),
        "pct_41_44h":        round(p4144, 2),
        "pct_acima_44h":     round(p_44, 1),
        "vab_r$_bi":         round(vab_rs / 1e9, 2),
        "produt_r$_hora":    round(produt_hora, 2),
        "parcela_afetada":   round(parcela_afetada, 4),
        "delta_h_pct":       round(DELTA_H_C * 100, 4),
        "delta_vab_r$_bi":   round(delta_vab_rs / 1e9, 3),
        "intensidade_pct":   round(intensidade_pct, 3),
    })

df_base = pd.DataFrame(registros)

print("\n── BASE ANALÍTICA SETORIAL 2023 (Formulação C) ──────────")
cols_view = ["setor", "qualidade_proxy", "h_media", "pct_41_44h",
             "vab_r$_bi", "produt_r$_hora", "delta_vab_r$_bi", "intensidade_pct"]
print(df_base[cols_view].to_string(index=False))

In [ ]:
df_incl = df_base[df_base["incluir"] == True].copy()
df_excl = df_base[df_base["incluir"] == False].copy()

delta_incl   = df_incl["delta_vab_r$_bi"].sum() * 1e9
delta_excl   = df_excl["delta_vab_r$_bi"].sum() * 1e9
delta_total  = delta_incl + delta_excl

pct_incl  = delta_incl  / pib_total_rs * 100
pct_excl  = delta_excl  / pib_total_rs * 100
pct_total = delta_total / pib_total_rs * 100

print("\n══ CENÁRIO BASE — Formulação C, sem ajuste de produtividade ══")
print(f"ΔH aplicado:            {DELTA_H_C*100:.4f}% (Formulação C)")
print(f"Faixa afetada:          41h < H ≤ 44h")
print()
print(f"Setores incluídos:")
print(f"  ΔVAB = R$ {delta_incl/1e9:.3f} bi  →  ΔPIB = {pct_incl:.4f}%")
print()
print(f"Setores excluídos (proxy imperfeita):")
print(f"  ΔVAB est. = R$ {delta_excl/1e9:.3f} bi  →  ΔPIB impl. = {pct_excl:.4f}%")
print()
print(f"── FAIXA DE INCERTEZA (atualizada) ─────────────────────")
print(f"  Limite inferior (setores incluídos):  {pct_incl:.4f}%")
print(f"  Limite superior (todos os setores):   {pct_total:.4f}%")
print(f"  Referência CNI:                       -0,7000%")
print()
print(f"Detalhamento (setores incluídos):")
print(df_incl[["setor", "pct_41_44h", "delta_vab_r$_bi", "intensidade_pct"]]
      .sort_values("delta_vab_r$_bi").to_string(index=False))

In [ ]:
df_base.to_csv(BASE_OUT, index=False)
df_incl[cols_view].to_csv(TAB_OUT, index=False)
print(f"\n✓ Base analítica salva: {BASE_OUT}")
print(f"✓ Tabela cenário base salva: {TAB_OUT}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

df_g  = df_incl.sort_values("delta_vab_r$_bi", ascending=True).copy()
df_g2 = df_incl.sort_values("intensidade_pct", ascending=True).copy()

qualidade_cores = {
    "direta":  "#2C6FBF",
    "parcial": "#F5A623",
    "ampla":   "#E84545",
}
cores1 = [qualidade_cores.get(q, "#888") for q in df_g["qualidade_proxy"]]
cores2 = [qualidade_cores.get(q, "#888") for q in df_g2["qualidade_proxy"]]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

bars1 = ax1.barh(df_g["setor"], df_g["delta_vab_r$_bi"], color=cores1, height=0.6)
ax1.axvline(0, color="black", linewidth=0.8)

for bar, (_, row) in zip(bars1, df_g.iterrows()):
    val = row["delta_vab_r$_bi"]
    if val < -10:
        x_pos = val + 0.5  
        align = 'left'
        c = 'white'
    else:
        x_pos = val - 0.5
        align = 'right'
        c = 'black'
        
    ax1.text(x_pos, bar.get_y() + bar.get_height()/2,
             f"R$ {val:.2f} bi",
             va="center", ha=align, fontsize=8.5, color=c, 
             fontweight='bold' if val < -10 else 'normal')

ax1.set_xlabel("Variação no VAB (R$ bilhões)", fontsize=10)
ax1.set_title("Impacto absoluto\n(contribuição para ΔPIB)", fontsize=11, loc="left")
ax1.set_xlim(df_g["delta_vab_r$_bi"].min() * 1.15, 0.5) 
bars2 = ax2.barh(df_g2["setor"], df_g2["intensidade_pct"], color=cores2, height=0.6)
ax2.axvline(0, color="black", linewidth=0.8)

for bar, (_, row) in zip(bars2, df_g2.iterrows()):
    val = row["intensidade_pct"]
    if val < -1.4:
        x_pos = val + 0.05
        align = 'left'
        c = 'white'
    else:
        x_pos = val - 0.05
        align = 'right'
        c = 'black'

    ax2.text(x_pos, bar.get_y() + bar.get_height()/2,
             f"{val:.3f}%",
             va="center", ha=align, fontsize=8.5, color=c)

ax2.set_xlabel("Variação relativa no VAB próprio (%)", fontsize=10)
ax2.set_title("Intensidade relativa\n(impacto / VAB do próprio setor)", fontsize=11, loc="left")
ax2.set_xlim(df_g2["intensidade_pct"].min() * 1.15, 0.05)

legenda = [
    mpatches.Patch(color="#2C6FBF", label="Proxy direta"),
    mpatches.Patch(color="#F5A623", label="Proxy parcial"),
    mpatches.Patch(color="#E84545", label="Proxy ampla"),
]
fig.legend(handles=legenda, loc="lower center", ncol=3, frameon=False,
           fontsize=9, bbox_to_anchor=(0.5, -0.05))

fig.suptitle(
    "Cenário base: impacto setorial da redução de jornada (44h→40h) — Brasil 2023\n"
    f"Formulação C (ΔH = {DELTA_H_C*100:.2f}%) · Faixa 41–44h · PNAD Contínua + CNT/IBGE",
    fontsize=12, y=1.01
)

fig.text(
    0.01, -0.08,
    f"ΔPIB (setores incluídos): {pct_incl:.4f}%  |  "
    f"Faixa de incerteza: [{pct_incl:.4f}%, {pct_total:.4f}%]  |  "
    f"Referência CNI: -0,700%  |  "
    "Formulação C: média ponderada uniforme de ΔH na faixa 41–44h.",
    fontsize=7.5, color="gray", transform=fig.transFigure
)

plt.tight_layout()
plt.savefig(DIR_FIGS / "11_cenario_base_setorial_v4.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gráfico 11v4 salvo com alinhamento corrigido.")